In [1]:
from transformers import AutoModel, AutoTokenizer

name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name).eval()

/Users/saif/Developer/work/sentinel/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1115 19:09:52.649000 52659 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10506.88it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bi

In [2]:
enc = tokenizer(["How many staff does the company employ?"], return_tensors="pt")
print(enc["input_ids"].shape)
print(tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print(enc["attention_mask"])

torch.Size([1, 10])
['[CLS]', 'how', 'many', 'staff', 'does', 'the', 'company', 'employ', '?', '[SEP]']
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [3]:
import torch

with torch.no_grad():
    out = model(**enc)
print(out.last_hidden_state.shape)

torch.Size([1, 10, 768])


In [4]:
def cls_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return hidden[:, 0]


def mean_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


def max_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    h = hidden.masked_fill(mask.unsqueeze(-1) == 0, -1e9)
    return h.max(1).values

In [5]:
import torch.nn.functional as functional

sentences = [
    "How many staff does the company employ?",
    "The firm had approximately 166,000 full-time employees. ",
    "The weather in Cupertino is mild.",
]
enc = tokenizer(sentences, padding=True, return_tensors="pt")
with torch.no_grad():
    hidden = model(**enc).last_hidden_state
mask = enc["attention_mask"]

for label, pool in [("CLS", cls_pool), ("mean", mean_pool), ("max", max_pool)]:
    v = functional.normalize(pool(hidden, mask), dim=-1)
    related = (v[0] @ v[1]).item()
    unrelated = (v[0] @ v[2]).item()
    print(
        f"{label:4} related={related:.3f} unrelated={unrelated:.3f} gap={related - unrelated:.3f}"
    )

CLS  related=0.849 unrelated=0.739 gap=0.110
mean related=0.644 unrelated=0.522 gap=0.122
max  related=0.859 unrelated=0.831 gap=0.028


In [6]:
from transformers import AutoModelForSequenceClassification

clf = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2).eval()
with torch.no_grad():
    logits = clf(**tokenizer(["Great quarter."], return_tensors="pt")).logits
print(logits.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11648.24it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

torch.Size([1, 2])
